In [1]:

import pandas as pd
import numpy as np
from langchain.document_loaders import PyPDFLoader
from langchain.document_loaders import PyPDFDirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
import uuid
import json
import ollama.client as client


splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    length_function=len,
    is_separator_regex=False
)


In [2]:
from transformers import pipeline

## Roberta based NER
# ner = pipeline("token-classification", model="2rtl3/mn-xlm-roberta-base-named-entity", aggregation_strategy="simple")
ner = pipeline("token-classification", model="dslim/bert-large-NER", aggregation_strategy="simple")


print("Number of parameters ->", ner.model.num_parameters()/1000000, "Mn")


Some weights of the model checkpoint at dslim/bert-large-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


Number of parameters -> 332.538889 Mn


In [3]:
def row2NamedEntities(row):
    # print(row)
    ner_results = ner(row['text'])
    metadata = {'chunk_id': row['chunk_id']}
    entities = []
    for result in ner_results:
        entities = entities + [{'name': result['word'], 'entity': result['entity_group'], **metadata}]
        
    return entities

def dfText2DfNE(dataframe):
    ## Takes a dataframe from the parsed data and returns dataframe with named entities. 
    ## The input dataframe must have a text and a chunk_id column. 

    ## Using swifter for parallelism
    ## 1. Calculate named entities for each row of the dataframe. 
    results = dataframe.apply(row2NamedEntities, axis=1)

    ## Flatten the list of lists to one single list of entities. 
    entities_list = [entity for sublist in results if sublist for entity in sublist]


    ## Remove all NaN entities
    entities_dataframe = pd.DataFrame(entities_list).replace(' ', np.nan)
    entities_dataframe = entities_dataframe.dropna(subset=['entity'])

    ## Count the number of occurances per chunk id
    entities_dataframe = entities_dataframe.groupby(['name', 'entity', 'chunk_id']).size().reset_index(name='count')

    return entities_dataframe

In [4]:
loader = PyPDFLoader("/home/kartikey/project/knowledge_graph/data_input/2407.04053v1.pdf")
# loader = PyPDFDirectoryLoader("./data/kesy1dd")

pages = loader.load_and_split(text_splitter=splitter)
len(pages)


245

In [5]:

rows = []
for page in pages:
    row = {'text': page.page_content, **page.metadata, 'chunk_id': uuid.uuid4().hex}
    rows += [row]

df = pd.DataFrame(rows)


In [6]:
dfne = dfText2DfNE(df)


In [7]:
df_ne = dfne.groupby(['name', 'entity']).agg({'count': 'sum', 'chunk_id': lambda x: ','.join(map(str, x))}).reset_index()

df_ne.sort_values(by='count', ascending=False).head(100).reset_index()

,index,name,entity,count,chunk_id
0,157,AI,MISC,108,"02bf5a13d3144fe5bdfdb73bfd1d76d1,0647702429984..."
1,818,S,PER,89,"07d24626643a41d1b8b8889997b2348a,0930c41ef9314..."
2,326,Edge AI,MISC,81,"0e3720de5d7a4958bff134c02c3f7dd0,14bc975e9b214..."
3,431,I,MISC,50,"0ecf542442414b988211a232d504421b,27f22a7f5e884..."
4,24,##T,MISC,41,"0ecf542442414b988211a232d504421b,27f22a7f5e884..."
...,...,...,...,...,...
95,828,S. Choudhury,PER,2,"32a4a82059d94ef2b364ba16f70d209d,d753042506fc4..."
96,364,G. Kaur,PER,2,"1024cffa94254ff197ecf229d5e92b3f,ef25a3ff6c7d4..."
97,217,C. Klein,PER,2,"91d7b1ed2168417b94e82ddb5078560b,cf7a941bc79f4..."
98,402,H. Hu,PER,2,"31833b8f155d4d50912702ff27979a5f,44db94be81064..."


In [8]:
pages[12].page_content

'bandwidth are signi\ue000cant constraints, such as in autonomous\ndriving, where decisions must be made in real time, or in\nhealthcare, where patient data must be processed quickly to\nprovide timely interventions [16]. The rise of edge AI is also\nfueled by advancements in hardware, such as more powerful\nand energy-ef\ue000cient processors, which make it feasible to run\nsophisticated AI models on devices like smartphones and IoT\nsensors [17].\nB. Motivation and Contributions\nThe motivation for integrating edge computing with AI\nis multifaceted, primarily driven by the imperative need for\nprocessing data in real time and navigating the inherent\nlimitations of centralized cloud computing systems [18]. As\nwe witness an exponential rise in the number of connected'

In [11]:
def extractConcepts(prompt: str, model='mistral-openorca:latest'):
    SYS_PROMPT = (
        "Your task is to extract the key entities mentioned in the users input.\n"
        "Entities may include - event, concept, person, place, object, document, organisation, artifact, misc, etc.\n"
        "Format your output as a list of json with the following structure.\n"
        "[{\n"
        "   \"entity\": The Entity string,\n"
        "   \"importance\": How important is the entity given the context on a scale of 1 to 5, 5 being the highest,\n"
        "   \"type\": Type of entity\n"
        "}, { }]"
    )
    response, context = client.generate(model_name=model, system=SYS_PROMPT, prompt=prompt)
    
    print("RAW RESPONSE FROM LLM:\n", response)  # Debug line
    
    try:
        return json.loads(response)
    except json.JSONDecodeError as e:
        print("Error decoding JSON:", e)
        return []  # Return empty if parsing fails


In [12]:
res = extractConcepts(prompt = pages[22].page_content)

Starting the generation process...
 [
  {
    "entity": "processing strain on IoT devices",
    "importance": 3,
    "type": "concept"
  },
  {
    "entity": "high transmission costs",
    "importance": 2,
    "type": "concept"
  },
  {
    "entity": "cloudlet resource use",
    "importance": 3,
    "type": "concept"
  },
  {
    "entity": "cyclic branch network",
    "importance": 2,
    "type": "artifact"
  },
  {
    "entity": "DL-based intelligent ofoading scheme",
    "importance": 3,
    "type": "artifact"
  },
  {
    "entity": "network edge computing power",
    "importance": 2,
    "type": "resource"
  },
  {
    "entity": "data trafc",
    "importance": 2,
    "type": "resource"
  },
  {
    "entity": "dual connectivity",
    "importance": 1,
    "type": "technology"
  },
  {
    "entity": "nonorthogonal multiple access computation ofoading systems",
    "importance": 2,
    "type": "technology"
  },
  {
    "entity": "Robust Neural Networks From Coded Classication (CoDNN

In [13]:
res

[{'entity': 'processing strain on IoT devices',
  'importance': 3,
  'type': 'concept'},
 {'entity': 'high transmission costs', 'importance': 2, 'type': 'concept'},
 {'entity': 'cloudlet resource use', 'importance': 3, 'type': 'concept'},
 {'entity': 'cyclic branch network', 'importance': 2, 'type': 'artifact'},
 {'entity': 'DL-based intelligent of\ue001oading scheme',
  'importance': 3,
  'type': 'artifact'},
 {'entity': 'network edge computing power',
  'importance': 2,
  'type': 'resource'},
 {'entity': 'data traf\ue000c', 'importance': 2, 'type': 'resource'},
 {'entity': 'dual connectivity', 'importance': 1, 'type': 'technology'},
 {'entity': 'nonorthogonal multiple access computation of\ue001oading systems',
  'importance': 2,
  'type': 'technology'},
 {'entity': 'Robust Neural Networks From Coded Classi\ue000cation (CoDNN)',
  'importance': 3,
  'type': 'artifact'},
 {'entity': 'multi-device collaborative pipelining processing of deep neural network (DNN) tasks',
  'importance': 